In [ ]:
deps_path = '/kaggle/input/datasets/nhhsag12/colpali-dependency'
!pip install --no-index --find-links {deps_path} --requirement {deps_path}/requirements.txt


In [ ]:
import os
import gc
import glob
import json
import pickle
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.notebook import tqdm

# ==============================================================================
# CONFIG — adjust paths to match your environment
# ==============================================================================

# Directory containing the split page-embedding pickle files produced by encoding notebook
PAGE_PKL_DIR   = "/kaggle/input/datasets/nhhsag12/dataset-encoded-page-colsmol"       # <-- change this

# ColPali model path (local checkpoint — used for ALL methods)
# COLPALI_BASE and COLPALI_LORA are already defined above; only LORA is needed
# for ColPali.from_pretrained() since it contains the merged weights + processor.

COLSMOL_BASE   = "/kaggle/input/models/nhhsag12/colsmolvlm-instruct-500m-base/pytorch/default/1"
COLSMOL_LORA   = "/kaggle/input/models/nhhsag12/colsmol-500m/pytorch/default/4"

# Dataset files
ANNOTATIONS_PATH = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_annotations.jsonl"
PAGES_PARQUET    = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_pages.parquet"

WORKING_DIR = "/kaggle/working"
os.makedirs(WORKING_DIR, exist_ok=True)

# Method config
TOPK_RATIOS        = [round(i * 0.1, 1) for i in range(1, 10)]   # 0.1 … 0.9
N_LAST_LAYERS_LIST = [4, 8, 16, 32]   # for Ours/Ablation method
NORMALIZE_MODES    = ['pre', 'post']   # for Ours/Ablation method
ATTN_N_LAYERS_LIST = [1, 2, 4, 8]     # for Attention Score method
KMEANS_ITERS       = 10               # for Spherical KMeans method
N_RANDOM_SEEDS     = 1                # for Random Pruning method (seeds to average)

print("Config loaded.")
print(f"TOPK_RATIOS       : {TOPK_RATIOS}")
print(f"N_LAST_LAYERS_LIST: {N_LAST_LAYERS_LIST}")
print(f"NORMALIZE_MODES   : {NORMALIZE_MODES}")
print(f"ATTN_N_LAYERS_LIST: {ATTN_N_LAYERS_LIST}")
print(f"KMEANS_ITERS      : {KMEANS_ITERS}")
print(f"N_RANDOM_SEEDS    : {N_RANDOM_SEEDS}")

In [ ]:
# ==============================================================================
# Load all page embedding pickle files and concatenate them.
#
# Encoding notebook output format:
#   pickle.dump((encoded_quote, quote_indices), f)
#   - encoded_quote  : list[np.ndarray]  shape (n_tokens, D)
#   - quote_indices  : list[int]  — row_index in MMDocIR_pages.parquet
#
# If the documents were split into multiple pkl files, we merge them here.
# ==============================================================================

pkl_files = sorted(glob.glob(os.path.join(PAGE_PKL_DIR, "*.pkl")))
print(f"Found {len(pkl_files)} page PKL file(s):")
for p in pkl_files:
    print(f"  {p}")

if not pkl_files:
    raise FileNotFoundError(f"No pkl files found in {PAGE_PKL_DIR}")

all_page_embeddings = []   # list[np.ndarray]  each shape (n_tokens, D)
all_page_indices    = []   # list[int]  parquet row index, parallel to embeddings

for pkl_path in pkl_files:
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)
    if isinstance(data, tuple) and len(data) == 2:
        embs, idxs = data
    elif isinstance(data, dict):
        embs = data.get('embeddings', data.get('encoded_quote', []))
        idxs = data.get('indices', data.get('quote_indices', list(range(len(embs)))))
    else:
        raise ValueError(f"Unknown pkl format in {pkl_path}")
    all_page_embeddings.extend(list(embs))
    all_page_indices.extend(list(idxs))
    print(f"  Loaded {len(embs)} page embeddings from {os.path.basename(pkl_path)}")

print(f"\nTotal page embeddings : {len(all_page_embeddings)}")
print(f"Embedding shape sample: {all_page_embeddings[0].shape}")

In [ ]:
from colpali_engine.models import ColIdefics3, ColIdefics3Processor
from peft import PeftModel

query_model = ColIdefics3.from_pretrained(
    COLSMOL_BASE,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
    attn_implementation="eager" # or eager
)
query_model = PeftModel.from_pretrained(
    query_model,
    COLSMOL_LORA
).eval()
query_processor = ColIdefics3Processor.from_pretrained(COLSMOL_LORA)


In [ ]:
def forward_with_attentions_colpali(model, inputs):
    """
    Run ColPali forward pass and capture per-layer attention weights via hooks.

    Tìm transformer decoder layers bằng cách duyệt toàn bộ submodule thay vì
    hardcode path — tránh AttributeError khi cấu trúc PaliGemma thay đổi theo
    phiên bản transformers.

    Returns:
      proj        : (B, S, dim)  — từ ColPali.forward(), giống Method 1
      raw_outputs : SimpleNamespace với .attentions = tuple of (B, H, S, S) tensors
    """
    from types import SimpleNamespace

    captured_attns = []
    hooks = []

    # Tìm GemmaModel.layers bằng cách duyệt submodules thay vì hardcode path.
    # Với PaliGemmaForConditionalGeneration, cấu trúc thực tế là:
    #   model.model  (PaliGemmaForConditionalGeneration)
    #     .model     (PaliGemmaModel)
    #       .language_model (GemmaForCausalLM)
    #         .model (GemmaModel)
    #           .layers (ModuleList)
    # Nhưng có thể khác nhau tùy phiên bản — nên tìm động.
    layers = None
    candidates = [
        # path đúng nhất theo class ColPali trong notebook này
        lambda m: m.model.model.language_model.model.layers,
        # fallback 1: một số phiên bản bỏ wrapper .model bên trong
        lambda m: m.model.model.language_model.layers,
        # fallback 2: PaliGemmaForConditionalGeneration không có language_model attr
        #             mà dùng .language_model trực tiếp trên .model
        lambda m: m.model.language_model.model.layers,
        lambda m: m.model.language_model.layers,
    ]
    for try_fn in candidates:
        try:
            layers = try_fn(model)
            break
        except AttributeError:
            continue

    # Nếu vẫn không tìm thấy: quét toàn bộ named_modules để tìm ModuleList chứa self_attn
    if layers is None:
        for name, mod in model.named_modules():
            if (isinstance(mod, torch.nn.ModuleList)
                    and len(mod) > 0
                    and hasattr(mod[0], 'self_attn')):
                layers = mod
                break

    if layers is None:
        raise RuntimeError(
            "Không tìm được transformer decoder layers trong model ColPali. "
            "Kiểm tra lại cấu trúc model bằng: print(model)"
        )

    def _attn_hook(module, inp, out):
        # GemmaAttention trả về (attn_output, attn_weights, past_key_value)
        # attn_weights là (B, H, S, S) khi output_attentions=True, ngược lại là None
        if isinstance(out, tuple) and len(out) > 1 and out[1] is not None:
            captured_attns.append(out[1].detach())

    for layer in layers:
        hooks.append(layer.self_attn.register_forward_hook(_attn_hook))

    try:
        with torch.no_grad():
            kwargs = dict(inputs)
            kwargs['output_attentions'] = True
            proj = model(**kwargs)   # ColPali.forward → (B, S, dim)
    finally:
        for h in hooks:
            h.remove()

    raw_outputs = SimpleNamespace(
        attentions=tuple(captured_attns) if captured_attns else None
    )
    return proj, raw_outputs

# ==============================================================================
# Helper: encode_query_live
#
# Encodes a single query string and returns:
#   proj        : (1, S, dim)  L2-normalised, masked  [float32 on device]
#   raw_outputs : ModelOutput  (contains .attentions for Methods 2 & 4)
#   inputs      : BatchEncoding (contains attention_mask, input_ids)
#   encode_ms   : wall-clock encoding time in milliseconds
#
# process_queries() chain (from ColPaliProcessor source):
#   process_queries([q])
#     → prepends query_prefix ("") + appends pad_token × 10
#     → calls process_texts()
#     → tokenizer(bos_token + text, return_tensors="pt", padding="longest")
# ==============================================================================

def encode_query_live(question: str, processor, model, device: str):
    """
    Tokenise `question` with process_queries() and run a forward pass that
    also captures attention weights.  Used by Methods 2 and 4.
    Returns (proj, raw_outputs, inputs, encode_ms).
    """
    inputs = processor.process_queries([question]).to(device)

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    with torch.no_grad():
        proj, raw_outputs = forward_with_attentions_colpali(model, inputs)

    torch.cuda.synchronize()
    encode_ms = (time.perf_counter() - t0) * 1000.0

    return proj, raw_outputs, inputs, encode_ms

In [ ]:
# DEBUG CELL — run this before the QA mapping cell
import json

print("Loading MMDocIR_pages.parquet...")
pages_df = pd.read_parquet(PAGES_PARQUET)
print(f"  Pages parquet: {len(pages_df)} rows, columns: {list(pages_df.columns)}")

# Build reverse lookup and per-doc lookup first (needed for debug checks)
parquet_row_to_embed_idx = {parquet_row: embed_idx
                            for embed_idx, parquet_row in enumerate(all_page_indices)}

embedded_rows = pages_df.iloc[all_page_indices].copy()
embedded_rows['embed_idx']     = list(range(len(all_page_indices)))
embedded_rows['join_doc_name'] = embedded_rows['doc_name'].str.replace('.pdf', '', regex=False)
embedded_rows['passage_id']    = embedded_rows['passage_id'].astype(str)

doc_page_lookup = {doc: grp for doc, grp in embedded_rows.groupby('join_doc_name')}
avail_docs      = set(doc_page_lookup.keys())

# 1. What does passage_id look like in the parquet?
print("=== pages_df sample ===")
print(pages_df[['doc_name', 'passage_id']].head(10).to_string())
print(f"\npassage_id dtype: {pages_df['passage_id'].dtype}")
print(f"passage_id sample values: {pages_df['passage_id'].iloc[:5].tolist()}")

# 2. What does the annotation's page_id look like?
print("\n=== annotation sample ===")
with open(ANNOTATIONS_PATH, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        doc_data = json.loads(line.strip())
        doc_name = doc_data['doc_name'].replace('.pdf', '')
        if doc_name in avail_docs:
            print(f"doc_name : {doc_name}")
            print(f"domain   : {doc_data.get('domain')}")
            for qa in doc_data['questions'][:3]:
                pid = qa.get('page_id')
                print(f"  Q: {qa['Q'][:60]}")
                print(f"  page_id: {repr(pid)}  type={type(pid).__name__}")
            break

# 3. Check doc_name overlap
print("\n=== doc_name overlap check ===")
annot_docs = set()
with open(ANNOTATIONS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        d = json.loads(line.strip())
        annot_docs.add(d['doc_name'].replace('.pdf', ''))
print(f"Annotation docs : {len(annot_docs)}")
print(f"Embedded docs   : {len(avail_docs)}")
print(f"Intersection    : {len(annot_docs & avail_docs)}")
print(f"Sample annot    : {sorted(annot_docs)[:3]}")
print(f"Sample embedded : {sorted(avail_docs)[:3]}")

In [ ]:
# ==============================================================================
# Cell 4 — Build Page → Document Mapping and QA Pairs
# ==============================================================================

print("\nBuilding QA pairs from annotations...")

qa_pairs = []
q_global = 0

with open(ANNOTATIONS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        try:
            doc_data = json.loads(line.strip())
        except Exception:
            continue

        doc_name = doc_data['doc_name'].replace('.pdf', '')
        domain   = doc_data.get('domain', 'General')

        if doc_name not in avail_docs:
            q_global += len(doc_data.get('questions', []))
            continue

        doc_pages = doc_page_lookup[doc_name]

        for qa in doc_data.get('questions', []):
            question_text = qa['Q']
            gt_page_ids   = qa.get('page_id', [])

            if not isinstance(gt_page_ids, list):
                gt_page_ids = [gt_page_ids]
            gt_page_ids_str = [str(p) for p in gt_page_ids]

            gt_embed_idxs = []
            for pid_str in gt_page_ids_str:
                matched = doc_pages[doc_pages['passage_id'] == pid_str]
                gt_embed_idxs.extend(matched['embed_idx'].tolist())

                if not matched.shape[0]:
                    matched_full = embedded_rows[
                        (embedded_rows['passage_id']    == pid_str) &
                        (embedded_rows['join_doc_name'] == doc_name)
                    ]
                    gt_embed_idxs.extend(matched_full['embed_idx'].tolist())

            if gt_embed_idxs:
                qa_pairs.append({
                    'question':         question_text,
                    'gt_embed_indices': list(set(gt_embed_idxs)),
                    'doc_name':         doc_name,
                    'domain':           domain,
                    # NOTE: no query_embed_idx — queries are encoded on-the-fly
                })

            q_global += 1

qa_pairs = [q for q in qa_pairs if q['gt_embed_indices']]
print(f"Total QA pairs built : {len(qa_pairs)}")
print(f"QA pairs after filter: {len(qa_pairs)}")

In [ ]:
# ==============================================================================
# Shared utilities: doc matrix builder, MaxSim, metrics, latency tracker
# ==============================================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


def build_doc_matrix(embeddings, device):
    """
    Convert list of np.ndarray embeddings to a padded (n_docs, max_len, D) tensor.
    Returns (doc_matrix, doc_mask).
    """
    arrays   = [torch.from_numpy(e).float() for e in embeddings]
    max_len  = max(a.shape[0] for a in arrays)
    D        = arrays[0].shape[1]
    n        = len(arrays)
    mat      = torch.zeros(n, max_len, D, dtype=torch.float32)
    mask     = torch.zeros(n, max_len, dtype=torch.bool)
    for i, a in enumerate(arrays):
        L = a.shape[0]
        mat[i, :L] = F.normalize(a, dim=-1)
        mask[i, :L] = True
    return mat.to(device), mask.to(device)


@torch.no_grad()
def fast_maxsim(q_norm, doc_matrix, doc_mask):
    """
    q_norm   : (N_q, D)   — query tokens, L2-normalized
    doc_matrix: (n_docs, max_len, D)
    doc_mask : (n_docs, max_len)  bool
    Returns  : (N_q, n_docs)  per-token MaxSim scores
    """
    sim = torch.einsum('qd,nld->qnl', q_norm, doc_matrix)   # (N_q, n_docs, max_len)
    sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
    return sim.max(dim=-1).values                             # (N_q, n_docs)


# ---------- Metrics ----------

def compute_ndcg(ranked, gt, k):
    dcg  = sum(1.0 / np.log2(r + 2) for r, i in enumerate(ranked[:k]) if i in gt)
    idcg = sum(1.0 / np.log2(r + 2) for r in range(min(len(gt), k)))
    return dcg / idcg if idcg > 0 else 0.0

def first_hit(top_k, gt):
    for r, i in enumerate(top_k):
        if i in gt: return r + 1
    return -1

def hit_metrics(top10, gt):
    h = first_hit(top10, gt)
    return {
        'r1':  int(h != -1 and h <= 1),
        'r5':  int(h != -1 and h <= 5),
        'r10': int(h != -1 and h <= 10),
        'n1':  float(compute_ndcg(top10, gt, 1)),
        'n5':  float(compute_ndcg(top10, gt, 5)),
        'n10': float(compute_ndcg(top10, gt, 10)),
    }

def _init_metric():
    return {'r1': 0, 'r5': 0, 'r10': 0, 'n1': 0.0, 'n5': 0.0, 'n10': 0.0, 'count': 0}

def _add_metric(dst, src):
    dst['r1']    += int(src['r1'])
    dst['r5']    += int(src['r5'])
    dst['r10']   += int(src['r10'])
    dst['n1']    += float(src['n1'])
    dst['n5']    += float(src['n5'])
    dst['n10']   += float(src['n10'])
    dst['count'] += 1

def _ensure(store, key):
    if key not in store: store[key] = _init_metric()
    return store[key]

def record(all_metrics, all_domain_metrics, key, m, domain):
    _add_metric(_ensure(all_metrics, key), m)
    if domain not in all_domain_metrics:
        all_domain_metrics[domain] = {}
    _add_metric(_ensure(all_domain_metrics[domain], key), m)


def print_summary(all_metrics, all_domain_metrics, method_keys, title=""):
    if title:
        print(f"\n{'='*60}\n{title}\n{'='*60}")
    print(f"{'Method':<35} {'R@1':>7} {'R@5':>7} {'R@10':>7} {'nDCG@10':>9}")
    print("-" * 65)
    for key in method_keys:
        if key not in all_metrics: continue
        m = all_metrics[key]; cnt = m['count'] or 1
        print(f"{key:<35} {m['r1']/cnt*100:6.2f}%  {m['r5']/cnt*100:6.2f}%  "
              f"{m['r10']/cnt*100:6.2f}%  {m['n10']/cnt:8.4f}")


# ---------- Latency Tracker ----------

class LatencyTracker:
    """
    Tracks per-ratio latency of MaxSim scoring AFTER pooling/pruning.

    Only measures the time for: MaxSim scoring + aggregation + top-k
    on the already-reduced multi-vector representation.
    Excludes model forward pass, pooling, and pruning computation.

    Usage:
        tracker = LatencyTracker("Hierarchical Ward Pool")
        tracker.add_ratio(ratio, score_ms)   # inside loop, per ratio
        tracker.report()                     # after loop
    """
    def __init__(self, method_name: str):
        self.name = method_name
        self.ratio_ms = {}   # ratio -> list[float]  pool+retrieve ms per query

    def add_ratio(self, ratio: float, score_ms: float):
        """Record scoring latency for a single (ratio, query) observation."""
        if ratio not in self.ratio_ms:
            self.ratio_ms[ratio] = []
        self.ratio_ms[ratio].append(score_ms)

    def report(self):
        if not self.ratio_ms:
            print(f"[{self.name}] No latency data collected.")
            return
        print(f"\n{'='*70}")
        print(f"Latency Report — {self.name}  (scoring only, post-pooling/pruning)")
        print(f"{'='*70}")
        print(f"  {'Ratio':<10} {'n':>6} {'avg ms':>10} {'p50 ms':>10} {'p95 ms':>10}")
        print(f"  {'-'*50}")
        for ratio in sorted(self.ratio_ms.keys()):
            vals = self.ratio_ms[ratio]
            n    = len(vals)
            avg  = np.mean(vals)
            p50  = np.percentile(vals, 50)
            p95  = np.percentile(vals, 95)
            print(f"  {ratio:<10.0%} {n:>6} {avg:>10.2f} {p50:>10.2f} {p95:>10.2f}")

    def to_dict(self):
        rows = []
        for ratio in sorted(self.ratio_ms.keys()):
            vals = self.ratio_ms[ratio]
            n    = len(vals)
            rows.append({
                'method':           self.name,
                'ratio':            ratio,
                'n_queries':        n,
                'avg_score_ms':     round(np.mean(vals), 3) if n else 0,
                'p50_score_ms':     round(np.percentile(vals, 50), 3) if n else 0,
                'p95_score_ms':     round(np.percentile(vals, 95), 3) if n else 0,
            })
        return rows


# ==============================================================================
# Spherical KMeans (cosine similarity) — Method 5
# Input : X (N, D) L2-normalized query token vectors
# Output: centroids (K, D) — mean-pool each cluster then re-normalize
# ==============================================================================

def spherical_kmeans(X, K, n_iters=KMEANS_ITERS):
    """
    Cluster N query token vectors into K representative centroids
    using cosine distance (spherical KMeans).

    Args:
        X      : (N, D)  L2-normalized query token vectors
        K      : int     number of clusters (representative tokens to keep)
        n_iters: int     number of Lloyd-style iterations

    Returns:
        centroids: (K, D)  L2-normalized cluster centroids
    """
    N, D = X.shape
    K = min(K, N)

    if K >= N:
        return X.clone()   # every token is its own representative

    # Initialise: pick K distinct tokens at random
    perm      = torch.randperm(N, device=X.device)
    centroids = X[perm[:K]].clone()   # (K, D)

    for _ in range(n_iters):
        # Assignment: cosine sim = dot product when X is normalised
        sim         = torch.mm(X, centroids.t())   # (N, K)
        cluster_ids = sim.argmax(dim=1)             # (N,)

        # Update: mean-pool members → re-normalize
        new_centroids = torch.zeros_like(centroids)
        for c in range(K):
            members = (cluster_ids == c).nonzero(as_tuple=True)[0]
            new_centroids[c] = (X[members].mean(dim=0)
                                if members.numel() > 0 else centroids[c])
        centroids = F.normalize(new_centroids, dim=-1)

    return centroids   # (K, D)


# ==============================================================================
# Random Token Pruning — Method 6
# Randomly sample round(M * ratio) content tokens; average over N_RANDOM_SEEDS.
# ==============================================================================

def random_prune_topk(q_norm, content_idx, doc_matrix, doc_mask,
                      topk_ratios, n_seeds=N_RANDOM_SEEDS):
    """
    For each ratio r keep k = round(M * r) randomly selected tokens.
    Scores are averaged over `n_seeds` independent random draws to reduce
    variance.

    Timing covers the full scoring work per ratio:
      random sampling + MaxSim + sum  (averaged over n_seeds).
    This matches how all other methods measure latency.

    Args:
        q_norm      : (M, D)   content token vectors, L2-normalized
        content_idx : (M,)     positions of content tokens in the full sequence
                               (unused here; kept for API symmetry)
        doc_matrix  : (n_docs, max_len, D)
        doc_mask    : (n_docs, max_len)  bool
        topk_ratios : list[float]  fractions of tokens to KEEP
        n_seeds     : int  number of random samples to average

    Returns:
        results : dict {ratio: scores (n_docs,)}
        timing  : dict {ratio: score_ms}   ms for sampling + MaxSim per ratio
    """
    M      = q_norm.shape[0]
    n_docs = doc_matrix.shape[0]
    device = q_norm.device

    if M == 0:
        zero = torch.zeros(n_docs, device=device)
        return {r: zero for r in topk_ratios}, {r: 0.0 for r in topk_ratios}

    results = {}
    timing  = {}

    for ratio in topk_ratios:
        k = max(1, round(M * ratio))
        k = min(k, M)

        # ── Time: random sample + MaxSim + sum (all scoring work) ────────
        torch.cuda.synchronize()
        t0 = time.perf_counter()

        if k == M:
            # Keep all tokens — no pruning needed; score as-is
            scores = fast_maxsim(q_norm, doc_matrix, doc_mask).sum(dim=0)
        else:
            accumulated = torch.zeros(n_docs, device=device, dtype=torch.float32)
            for _ in range(n_seeds):
                perm        = torch.randperm(M, device=device)[:k]
                pruned      = q_norm[perm]          # (k, D)
                accumulated += fast_maxsim(pruned, doc_matrix, doc_mask).sum(dim=0)
            scores = accumulated / n_seeds

        torch.cuda.synchronize()
        timing[ratio] = (time.perf_counter() - t0) * 1000.0
        # ─────────────────────────────────────────────────────────────────

        results[ratio] = scores

    return results, timing


print("Utility functions ready.")

In [ ]:
# ==============================================================================
# METHOD 1 — Traditional MaxSim
# Queries encoded live with ColPali (plain forward, no attention capture).
# ==============================================================================

print(">>> METHOD 1: Traditional MaxSim")

trad_metrics        = {}
trad_domain_metrics = {}
trad_query_rows     = []
trad_latency        = LatencyTracker("Traditional MaxSim")

METHOD_KEYS_TRAD = ['traditional']

# Build full doc matrix from all page embeddings
print(f"Building doc matrix from {len(all_page_embeddings)} page embeddings...")
doc_matrix, doc_mask = build_doc_matrix(all_page_embeddings, device)
n_docs = doc_matrix.shape[0]
print(f"Doc matrix shape: {doc_matrix.shape}")

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Traditional MaxSim")

for q_idx, item in pbar:
    question = item['question']
    gt_set   = set(item['gt_embed_indices'])
    domain   = item['domain']

    # ── Encode query live ──────────────────────────────────────────────
    q_inputs = query_processor.process_queries([question]).to(device)

    with torch.no_grad():
        q_proj = query_model(**q_inputs)   # (1, S, dim)  — plain forward

    attn_mask = q_inputs['attention_mask'][0]                       # (S,)
    trad_idx  = torch.where(attn_mask > 0)[0]
    q_emb     = q_proj[0][trad_idx].float()                         # (N, dim)
    q_norm    = F.normalize(q_emb, dim=-1)

    # ── Retrieval (timed — this is the 100% baseline) ───────────────────
    torch.cuda.synchronize()
    t_score_start = time.perf_counter()

    M      = fast_maxsim(q_norm, doc_matrix, doc_mask)               # (N, n_docs)
    scores = M.sum(dim=0)
    top10  = torch.topk(scores, min(10, n_docs)).indices.cpu().tolist()

    torch.cuda.synchronize()
    score_ms = (time.perf_counter() - t_score_start) * 1000.0
    trad_latency.add_ratio(1.0, score_ms)

    m = hit_metrics(top10, gt_set)
    record(trad_metrics, trad_domain_metrics, 'traditional', m, domain)

    trad_query_rows.append({
        'query_id':       q_idx,
        'doc_name':       item['doc_name'],
        'domain':         domain,
        'question':       question,
        'trad_r@1':       m['r1'],
        'trad_r@5':       m['r5'],
        'trad_r@10':      m['r10'],
        'trad_ndcg@1':    round(m['n1'],  4),
        'trad_ndcg@5':    round(m['n5'],  4),
        'trad_ndcg@10':   round(m['n10'], 4),
    })

print_summary(trad_metrics, trad_domain_metrics, METHOD_KEYS_TRAD,
              title="Traditional MaxSim Results")
trad_latency.report()

pd.DataFrame(trad_query_rows).to_csv(
    os.path.join(WORKING_DIR, "traditional_queries.csv"), index=False)
print("\n✅ Saved: traditional_queries.csv")

In [ ]:
# ==============================================================================
# SVD / importance helpers
# ==============================================================================

SVD_RANK_REMOVE  = 1
TEMPERATURE_OURS = 0.5   # softplus temperature


def remove_sink_components_batch(attn_heads, k):
    try:
        U, S, Vh = torch.linalg.svd(attn_heads, full_matrices=False)
        k = min(k, S.shape[-1])
        sink = (U[..., :k] * S[:, :k].unsqueeze(1)) @ Vh[:, :k, :]
        return (attn_heads - sink).clamp(min=0.0)
    except Exception:
        return attn_heads


def compute_svd_importance_softplus(attentions, content_mask, layer_weights, k):
    """
    attentions  : list of (B, H, Sq, Sk) tensors — one per layer
    content_mask: (B, S) float tensor
    layer_weights: (n_layers,) tensor, exponentially increasing
    Returns     : (B, S) importance tensor
    """
    device_    = content_mask.device
    B, S       = content_mask.shape
    importance = torch.zeros(B, S, device=device_)

    for i, attn in enumerate(attentions):
        attn      = attn.float().to(device_)
        B_, H, Sq, Sk = attn.shape
        attn_flat = attn.view(B_ * H, Sq, Sk)
        cleaned   = remove_sink_components_batch(attn_flat, k)
        cleaned   = cleaned.view(B_, H, Sq, Sk)

        layer_imp = cleaned.sum(dim=2).mean(dim=1)          # (B, S)
        layer_imp = layer_imp * content_mask
        layer_imp = layer_imp / layer_imp.max(dim=-1, keepdim=True).values.clamp(min=1e-8)
        importance += layer_weights[i] * layer_imp

    importance = importance * content_mask
    importance = F.softplus(importance / TEMPERATURE_OURS)
    importance = importance * content_mask
    return importance


def build_content_mask_qwen(inputs, processor):
    """Mask out padding and special tokens from the attention mask."""
    attn_mask = inputs["attention_mask"]
    input_ids = inputs.get("input_ids", None)
    if input_ids is None:
        return attn_mask.float()

    tok = getattr(processor, 'tokenizer', processor)
    special_ids = set()
    for attr in ['pad_token_id', 'bos_token_id', 'eos_token_id',
                 'unk_token_id', 'sep_token_id', 'cls_token_id']:
        tid = getattr(tok, attr, None)
        if tid is not None:
            special_ids.add(int(tid))
    if hasattr(tok, 'added_tokens_encoder'):
        for _, tid in tok.added_tokens_encoder.items():
            special_ids.add(int(tid))
    if not special_ids:
        return attn_mask.float()

    special_tensor = torch.tensor(list(special_ids), device=input_ids.device)
    is_special     = (input_ids.unsqueeze(-1) == special_tensor).any(dim=-1)
    return attn_mask.float() * (~is_special).float()


def build_cluster_pool_vectors_ours(
    q_norm_kept, q_norm_disc,
    imp_kept, imp_disc,
    normalize_mode='pre',
):
    """
    Pooling step ONLY: assign discarded tokens to nearest kept token,
    then create weighted-average pooled vectors per cluster.

    Returns (pooled_vecs, pooled_weights) where:
      pooled_vecs   : (K, D) tensor — one pooled vector per cluster
      pooled_weights: (K,) tensor   — total importance weight per cluster
    If no discarded tokens, returns empty tensors.
    """
    P = q_norm_disc.shape[0]
    if P == 0:
        device_ = q_norm_kept.device
        D = q_norm_kept.shape[1]
        return torch.zeros(0, D, device=device_), torch.zeros(0, device=device_)

    sim_assign  = torch.mm(q_norm_disc, q_norm_kept.t())
    cluster_ids = sim_assign.argmax(dim=-1)

    pooled_vecs    = []
    pooled_weights = []
    for c in cluster_ids.unique():
        members  = (cluster_ids == c).nonzero(as_tuple=True)[0]
        w        = imp_disc[members]
        w_sum    = w.sum().clamp(min=1e-8)
        w_norm   = w / w_sum
        pool_vec = (q_norm_disc[members] * w_norm.unsqueeze(-1)).sum(dim=0)
        if normalize_mode == 'pre':
            pool_vec = F.normalize(pool_vec.unsqueeze(0), dim=-1).squeeze(0)
        pooled_vecs.append(pool_vec)
        pooled_weights.append(w_sum)

    return torch.stack(pooled_vecs), torch.stack(pooled_weights)


def make_ablation_keys_ours():
    keys = []
    for n in N_LAST_LAYERS_LIST:
        for mode in NORMALIZE_MODES:
            for r in TOPK_RATIOS:
                tag = f"L{n}_norm{mode}_r{int(r*100)}"
                keys.append(f"{tag}_trad")
                keys.append(f"{tag}_weighted")
    return keys

ABLATION_KEYS_OURS = make_ablation_keys_ours()

In [ ]:
# ==============================================================================
# Cell 7b — METHOD 2: Ours — SVD Importance + Cluster Pool (v12-Ablation)
# ==============================================================================

print(">>> METHOD 2: Ours — SVD Importance + Cluster Pool (v12-Ablation)")

ours_metrics        = {}
ours_domain_metrics = {}
ours_query_rows     = []
ours_latency        = LatencyTracker("Ours (SVD+ClusterPool)")

# doc_matrix / doc_mask / n_docs already built in Cell 6 — reuse them

print(f"Running ablation over {len(qa_pairs)} queries")
print(f"  n_last_layers : {N_LAST_LAYERS_LIST}")
print(f"  normalize_mode: {NORMALIZE_MODES}")
print(f"  topk_ratios   : {TOPK_RATIOS}")

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Ours (SVD+ClusterPool)")

for q_idx, item in pbar:
    question = item['question']
    gt_set   = set(item['gt_embed_indices'])
    domain   = item['domain']

    # ── Encode query live (with attentions) ───────────────────────────
    proj, raw_outputs, q_inputs, encode_ms = encode_query_live(
        question, query_processor, query_model, device
    )

    attn_mask_1d    = q_inputs['attention_mask'][0].float()
    content_mask_1d = build_content_mask_qwen(q_inputs, query_processor)[0].float()

    trad_idx   = torch.where(attn_mask_1d   > 0)[0]
    method_idx = torch.where(content_mask_1d > 0)[0]
    if trad_idx.numel() == 0:
        continue
    if method_idx.numel() == 0:
        method_idx = trad_idx

    content_mask_2d = content_mask_1d.unsqueeze(0)
    embed_cache = {}   # n_layers -> (proj_1d, imp_1d)

    for n_layers in N_LAST_LAYERS_LIST:
        if raw_outputs.attentions is not None and len(raw_outputs.attentions) > 0:
            attn_list = list(raw_outputs.attentions[-n_layers:])
            n_actual  = len(attn_list)
        else:
            attn_list = []
            n_actual  = 0

        layer_weights = torch.exp(
            torch.linspace(0, 1, max(n_actual, 1), device=device)
        )
        layer_weights /= layer_weights.sum()

        if n_actual > 0:
            importance = compute_svd_importance_softplus(
                attn_list, content_mask_2d, layer_weights, SVD_RANK_REMOVE
            )
        else:
            importance = content_mask_2d.clone()

        importance = (importance * content_mask_2d)[0]
        embed_cache[n_layers] = (proj[0].float(), importance)

    query_row = {
        'query_id':      q_idx,
        'doc_name':      item['doc_name'],
        'domain':        domain,
        'question':      question,
    }

    # Ablation: n_last_layers × normalize_mode × ratio × scoring_mode
    for n_layers in N_LAST_LAYERS_LIST:
        q_embed, q_importance = embed_cache[n_layers]

        q_method_norm  = F.normalize(q_embed[method_idx].float(), dim=-1)

        imp_valid      = q_importance[method_idx].float()
        n_method       = method_idx.numel()
        sorted_imp_idx = torch.argsort(imp_valid, descending=True)

        for mode in NORMALIZE_MODES:
            for r in TOPK_RATIOS:
                tag    = f"L{n_layers}_norm{mode}_r{int(r*100)}"
                n_keep = max(1, int(n_method * r))

                # ── Pooling/pruning (NOT timed) ────────────────────
                kept_idx = sorted_imp_idx[:n_keep]
                disc_idx = sorted_imp_idx[n_keep:]

                q_kept   = q_method_norm[kept_idx]
                q_disc   = q_method_norm[disc_idx]
                imp_kept = imp_valid[kept_idx]
                imp_disc = imp_valid[disc_idx]

                pooled_vecs, pooled_weights = build_cluster_pool_vectors_ours(
                    q_kept, q_disc, imp_kept, imp_disc,
                    normalize_mode=mode,
                )
                total_disc_imp = imp_disc.sum().item()
                pool_frac = total_disc_imp / imp_valid.sum().clamp(min=1e-8).item()

                # Combine kept + pooled into a single set of query vectors
                if pooled_vecs.shape[0] > 0:
                    all_q_vecs = torch.cat([q_kept, pooled_vecs], dim=0)
                    all_q_weights = torch.cat([
                        imp_kept,
                        pooled_weights * pool_frac,
                    ])
                    n_kept_vecs = q_kept.shape[0]
                else:
                    all_q_vecs = q_kept
                    all_q_weights = imp_kept
                    n_kept_vecs = q_kept.shape[0]

                # ── Time MaxSim scoring on kept + pooled vectors ───
                torch.cuda.synchronize()
                t_score_start = time.perf_counter()

                M_all    = fast_maxsim(all_q_vecs, doc_matrix, doc_mask)

                # trad scoring: sum of kept MaxSim + sum of pooled MaxSim * pool_frac
                M_kept_part   = M_all[:n_kept_vecs]
                M_pooled_part = M_all[n_kept_vecs:]
                sc_trad = M_kept_part.sum(dim=0)
                if M_pooled_part.shape[0] > 0:
                    sc_trad = sc_trad + (M_pooled_part * pooled_weights.unsqueeze(-1)).sum(dim=0) / max(total_disc_imp, 1e-8) * pool_frac
                top10_trad = torch.topk(sc_trad, min(10, n_docs)).indices.cpu().tolist()

                # weighted scoring
                all_w_n  = all_q_weights / all_q_weights.sum().clamp(min=1e-8)
                sc_wt    = (M_all * all_w_n.unsqueeze(-1)).sum(dim=0)
                top10_wt = torch.topk(sc_wt, min(10, n_docs)).indices.cpu().tolist()

                torch.cuda.synchronize()
                score_ms = (time.perf_counter() - t_score_start) * 1000.0
                ours_latency.add_ratio(r, score_ms)
                # ────────────────────────────────────────────────────

                m_trad     = hit_metrics(top10_trad, gt_set)
                key_trad   = f"{tag}_trad"
                record(ours_metrics, ours_domain_metrics, key_trad, m_trad, domain)
                query_row.update({
                    f'{key_trad}_r@1':    m_trad['r1'],
                    f'{key_trad}_r@5':    m_trad['r5'],
                    f'{key_trad}_r@10':   m_trad['r10'],
                    f'{key_trad}_ndcg@10':round(m_trad['n10'], 4),
                })

                m_wt       = hit_metrics(top10_wt, gt_set)
                key_wt     = f"{tag}_weighted"
                record(ours_metrics, ours_domain_metrics, key_wt, m_wt, domain)
                query_row.update({
                    f'{key_wt}_r@1':    m_wt['r1'],
                    f'{key_wt}_r@5':    m_wt['r5'],
                    f'{key_wt}_r@10':   m_wt['r10'],
                    f'{key_wt}_ndcg@10':round(m_wt['n10'], 4),
                })

    ours_query_rows.append(query_row)

    if q_idx % 50 == 0:
        gc.collect()
        torch.cuda.empty_cache()

print_summary(ours_metrics, ours_domain_metrics,
              ABLATION_KEYS_OURS,
              title="Ours — SVD Importance + Cluster Pool")
ours_latency.report()

pd.DataFrame(ours_query_rows).to_csv(
    os.path.join(WORKING_DIR, "ours_ablation_queries.csv"), index=False)
print("\n✅ Saved: ours_ablation_queries.csv")

In [ ]:
# ==============================================================================
# METHOD 3 — Hierarchical: Ward agglomerative token pooling
# ==============================================================================

print(">>> METHOD 3: Hierarchical Ward Token Pooling")


def _ward_distance_matrix(cents, sizes):
    device_  = cents.device
    sim      = torch.matmul(cents, cents.t()).clamp(-1.0, 1.0)
    sq_dist  = 2.0 * (1.0 - sim)
    ni = sizes.float().unsqueeze(1)
    nj = sizes.float().unsqueeze(0)
    w  = (ni * nj) / (ni + nj)
    ward = w * sq_dist
    mask = torch.ones(cents.shape[0], cents.shape[0], dtype=torch.bool, device=device_).tril()
    ward.masked_fill_(mask, float('inf'))
    return ward


def ward_pool(vecs, n_clusters):
    """vecs: (N, D) L2-normalized. Returns centroids (K, D) L2-normalized."""
    N = vecs.shape[0]
    if N <= n_clusters:
        return vecs.clone()

    device_ = vecs.device
    sums    = vecs.clone().float()
    sizes   = torch.ones(N, device=device_, dtype=torch.float32)
    cents   = F.normalize(sums, dim=-1)
    active  = torch.ones(N, dtype=torch.bool, device=device_)
    C       = N

    while C > n_clusters:
        active_idx = active.nonzero(as_tuple=True)[0]
        c_act      = cents[active_idx]
        s_act      = sizes[active_idx]
        ward       = _ward_distance_matrix(c_act, s_act)
        flat_idx   = ward.argmin().item()
        n_act      = active_idx.shape[0]
        ai, aj     = flat_idx // n_act, flat_idx % n_act
        gi, gj     = active_idx[ai].item(), active_idx[aj].item()
        sums[gi]   = sums[gi] + sums[gj]
        sizes[gi]  = sizes[gi] + sizes[gj]
        cents[gi]  = F.normalize(sums[gi].unsqueeze(0), dim=-1).squeeze(0)
        active[gj] = False
        C -= 1

    final_idx = active.nonzero(as_tuple=True)[0]
    return cents[final_idx]


def ward_pool_scores_all_ratios(q_norm, doc_matrix, doc_mask, topk_ratios):
    """
    Returns (results, timing) where:
      results : dict[ratio -> scores tensor]
      timing  : dict[ratio -> score_ms]  (MaxSim scoring only, excludes ward pooling)
    """
    N       = q_norm.shape[0]
    results = {}
    timing  = {}
    prev_vecs, prev_C = q_norm.clone(), N

    for ratio in sorted(topk_ratios, reverse=True):
        # ── Ward pooling (NOT timed) ──────────────────────────
        target_C = max(1, round(N * ratio))
        if target_C >= prev_C:
            centroids = prev_vecs
        else:
            centroids = ward_pool(prev_vecs, target_C)
            prev_vecs, prev_C = centroids, centroids.shape[0]

        # ── Time ONLY MaxSim scoring after pooling ────────────
        torch.cuda.synchronize()
        t0 = time.perf_counter()

        M_c            = fast_maxsim(centroids, doc_matrix, doc_mask)
        results[ratio] = M_c.sum(0)

        torch.cuda.synchronize()
        timing[ratio] = (time.perf_counter() - t0) * 1000.0

    return results, timing


# ---------- Keys ----------
METHOD_KEYS_HIER = ['traditional'] + [f"hier_r{int(r*100)}" for r in TOPK_RATIOS]

hier_metrics        = {}
hier_domain_metrics = {}
hier_query_rows     = []
hier_latency        = LatencyTracker("Hierarchical Ward Pool")

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Hierarchical Ward Pool")

for q_idx, item in pbar:
    question = item['question']
    gt_set   = set(item['gt_embed_indices'])
    domain   = item['domain']

    # ── Encode query live ──────────────────────────────────────────────
    q_inputs = query_processor.process_queries([question]).to(device)

    with torch.no_grad():
        q_proj = query_model(**q_inputs)   # plain forward

    attn_mask = q_inputs['attention_mask'][0]
    trad_idx  = torch.where(attn_mask > 0)[0]
    q_emb     = q_proj[0][trad_idx].float()
    q_norm    = F.normalize(q_emb, dim=-1)

    # ── Retrieval ──────────────────────────────────────────────────────

    # Baseline traditional
    M_trad     = fast_maxsim(q_norm, doc_matrix, doc_mask)
    trad_sc    = M_trad.sum(dim=0)
    top10_trad = torch.topk(trad_sc, min(10, n_docs)).indices.cpu().tolist()
    m_trad     = hit_metrics(top10_trad, gt_set)
    record(hier_metrics, hier_domain_metrics, 'traditional', m_trad, domain)

    # Ward hierarchical pooling — all ratios, with per-ratio timing
    ratio_scores, ratio_timing = ward_pool_scores_all_ratios(q_norm, doc_matrix, doc_mask, TOPK_RATIOS)

    query_row = {
        'query_id': q_idx, 'doc_name': item['doc_name'],
        'domain': domain,  'question': question,
        'trad_r@1': m_trad['r1'],   'trad_r@5': m_trad['r5'],
        'trad_r@10': m_trad['r10'], 'trad_ndcg@10': round(m_trad['n10'], 4),
    }

    for r in TOPK_RATIOS:
        key   = f"hier_r{int(r*100)}"
        top10 = torch.topk(ratio_scores[r], min(10, n_docs)).indices.cpu().tolist()
        m     = hit_metrics(top10, gt_set)
        record(hier_metrics, hier_domain_metrics, key, m, domain)
        hier_latency.add_ratio(r, ratio_timing[r])
        query_row.update({
            f'{key}_r@1':     m['r1'],
            f'{key}_r@5':     m['r5'],
            f'{key}_r@10':    m['r10'],
            f'{key}_ndcg@10': round(m['n10'], 4),
        })

    hier_query_rows.append(query_row)

print_summary(hier_metrics, hier_domain_metrics, METHOD_KEYS_HIER,
              title="Hierarchical Ward Pooling Results")
hier_latency.report()

pd.DataFrame(hier_query_rows).to_csv(
    os.path.join(WORKING_DIR, "hierarchical_queries.csv"), index=False)
print("\n✅ Saved: hierarchical_queries.csv")

In [ ]:
# ==============================================================================
# METHOD 4 — Attention Score Token Pruning (v13)
#
# Per-token importance = column-sum of attention weights across all heads/layers.
# Queries are encoded live with forward_with_attentions; latency is measured.
# ==============================================================================

print(">>> METHOD 4: Attention Score Token Pruning (Lassance et al. 2021)")

# ---------- Keys ----------
METHOD_KEYS_ATTN = ['traditional']
for n in ATTN_N_LAYERS_LIST:
    for r in TOPK_RATIOS:
        METHOD_KEYS_ATTN += [
            f"attn_L{n}_r{int(r*100)}_trad",
            f"attn_L{n}_r{int(r*100)}_weighted",
        ]

attn_metrics        = {}
attn_domain_metrics = {}
attn_query_rows     = []
attn_latency        = LatencyTracker("Attention Score Pruning")

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Attention Score Pruning")

for q_idx, item in pbar:
    question = item['question']
    gt_set   = set(item['gt_embed_indices'])
    domain   = item['domain']

    # ── Encode query live (with attentions) ───────────────────────────
    proj, raw_outputs, q_inputs, encode_ms = encode_query_live(
        question, query_processor, query_model, device
    )

    attn_mask_1d = q_inputs['attention_mask'][0].float()
    trad_idx     = torch.where(attn_mask_1d > 0)[0]
    N            = trad_idx.numel()

    q_emb  = proj[0][trad_idx].float()
    q_norm = F.normalize(q_emb, dim=-1)

    # Baseline (full tokens)
    M_full     = fast_maxsim(q_norm, doc_matrix, doc_mask)
    trad_sc    = M_full.sum(dim=0)
    top10_trad = torch.topk(trad_sc, min(10, n_docs)).indices.cpu().tolist()
    m_trad     = hit_metrics(top10_trad, gt_set)
    record(attn_metrics, attn_domain_metrics, 'traditional', m_trad, domain)

    query_row = {
        'query_id': q_idx, 'doc_name': item['doc_name'],
        'domain': domain,  'question': question,
        'trad_r@1': m_trad['r1'],   'trad_r@5': m_trad['r5'],
        'trad_r@10': m_trad['r10'], 'trad_ndcg@10': round(m_trad['n10'], 4),
    }

    for n_layers in ATTN_N_LAYERS_LIST:
        # Compute attention column-sum importance from the last n_layers attention tensors
        # i(t) = Σ_h Σ_j A_{h,j,t}  (sum of attention received by token t, Lassance 2021)
        if raw_outputs.attentions is not None and len(raw_outputs.attentions) >= 1:
            attn_subset = list(raw_outputs.attentions[-n_layers:])
            # Each element: (1, H, Sq, Sk)  → sum over query dim → (1, H, Sk) → mean heads → (1, Sk)
            imp_2d = torch.zeros(1, q_inputs['attention_mask'].shape[1], device=device)
            for attn_layer in attn_subset:
                col_sum = attn_layer.float().sum(dim=2).mean(dim=1)  # (1, S)
                imp_2d += col_sum
            imp_full = imp_2d[0]                                      # (S,)
            imp      = imp_full[trad_idx]                             # (N,)
        else:
            imp = torch.ones(N, device=device)

        sorted_imp_idx = torch.argsort(imp, descending=True)

        for r in TOPK_RATIOS:
            n_keep   = max(1, int(N * r))

            # ── Pruning (NOT timed) ────────────────────────────
            kept_idx = sorted_imp_idx[:n_keep]
            imp_kept = imp[kept_idx]
            q_kept   = q_norm[kept_idx]                          # pruned vectors

            # ── Time MaxSim scoring on pruned vectors ──────────
            torch.cuda.synchronize()
            t_score_start = time.perf_counter()

            M_kept   = fast_maxsim(q_kept, doc_matrix, doc_mask) # (n_keep, n_docs)
            sc_trad  = M_kept.sum(dim=0)
            top_t    = torch.topk(sc_trad, min(10, n_docs)).indices.cpu().tolist()

            imp_k_n  = imp_kept / imp_kept.sum().clamp(min=1e-8)
            sc_w     = (M_kept * imp_k_n.unsqueeze(-1)).sum(dim=0)
            top_w    = torch.topk(sc_w, min(10, n_docs)).indices.cpu().tolist()

            torch.cuda.synchronize()
            score_ms = (time.perf_counter() - t_score_start) * 1000.0
            attn_latency.add_ratio(r, score_ms)
            # ────────────────────────────────────────────────────

            m_t      = hit_metrics(top_t, gt_set)
            key_t    = f"attn_L{n_layers}_r{int(r*100)}_trad"
            record(attn_metrics, attn_domain_metrics, key_t, m_t, domain)
            query_row.update({f'{key_t}_r@1': m_t['r1'], f'{key_t}_r@5': m_t['r5'],
                              f'{key_t}_r@10': m_t['r10'], f'{key_t}_ndcg@10': round(m_t['n10'], 4)})

            m_w      = hit_metrics(top_w, gt_set)
            key_w    = f"attn_L{n_layers}_r{int(r*100)}_weighted"
            record(attn_metrics, attn_domain_metrics, key_w, m_w, domain)
            query_row.update({f'{key_w}_r@1': m_w['r1'], f'{key_w}_r@5': m_w['r5'],
                              f'{key_w}_r@10': m_w['r10'], f'{key_w}_ndcg@10': round(m_w['n10'], 4)})

    attn_query_rows.append(query_row)

print_summary(attn_metrics, attn_domain_metrics,
              ['traditional'] + [f"attn_L1_r{int(r*100)}_trad" for r in TOPK_RATIOS],
              title="Attention Score Pruning (L=1, trad scoring)")
attn_latency.report()

pd.DataFrame(attn_query_rows).to_csv(
    os.path.join(WORKING_DIR, "attention_queries.csv"), index=False)
print("\n✅ Saved: attention_queries.csv")

In [ ]:
# ==============================================================================
# METHOD 5 — Spherical KMeans Token Pooling
#
# Groups the N query tokens into K = max(1, int(N * ratio)) clusters using
# spherical KMeans (cosine distance).  Each cluster is represented by its
# L2-normalised mean-pooled centroid.  MaxSim is then run with K centroid
# vectors instead of N raw token vectors.
#
# References: PLAID / ColBERT v2 cluster-pruning idea adapted for query-side.
# Queries are encoded live with the plain forward pass (no attentions needed).
# ==============================================================================

print(">>> METHOD 5: Spherical KMeans Token Pooling")

# ---------- Keys ----------
METHOD_KEYS_KMEANS = ['traditional'] + [f"kmeans_r{int(r*100)}" for r in TOPK_RATIOS]

kmeans_metrics        = {}
kmeans_domain_metrics = {}
kmeans_query_rows     = []
kmeans_latency        = LatencyTracker("Spherical KMeans Pool")

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Spherical KMeans Pool")

for q_idx, item in pbar:
    question = item['question']
    gt_set   = set(item['gt_embed_indices'])
    domain   = item['domain']

    # ── Encode query live (plain forward — no attentions needed) ──────────
    q_inputs = query_processor.process_queries([question]).to(device)

    with torch.no_grad():
        q_proj = query_model(**q_inputs)   # (1, S, dim)

    attn_mask = q_inputs['attention_mask'][0]
    trad_idx  = torch.where(attn_mask > 0)[0]
    q_emb     = q_proj[0][trad_idx].float()
    q_norm    = F.normalize(q_emb, dim=-1)          # (N, D)
    N         = q_norm.shape[0]

    # ── Baseline (full tokens) ────────────────────────────────────────────
    M_full     = fast_maxsim(q_norm, doc_matrix, doc_mask)
    trad_sc    = M_full.sum(dim=0)
    top10_trad = torch.topk(trad_sc, min(10, n_docs)).indices.cpu().tolist()
    m_trad     = hit_metrics(top10_trad, gt_set)
    record(kmeans_metrics, kmeans_domain_metrics, 'traditional', m_trad, domain)

    query_row = {
        'query_id': q_idx, 'doc_name': item['doc_name'],
        'domain': domain,  'question': question,
        'trad_r@1': m_trad['r1'],   'trad_r@5': m_trad['r5'],
        'trad_r@10': m_trad['r10'], 'trad_ndcg@10': round(m_trad['n10'], 4),
    }

    # ── Spherical KMeans pooling — all ratios ─────────────────────────────
    for r in TOPK_RATIOS:
        K = max(1, int(N * r))

        # ── KMeans pooling (NOT timed) ────────────────────────────
        centroids = spherical_kmeans(q_norm, K)       # (K, D)

        # ── Time MaxSim scoring on centroids ──────────────────────
        torch.cuda.synchronize()
        t0 = time.perf_counter()

        M_k   = fast_maxsim(centroids, doc_matrix, doc_mask)  # (K, n_docs)
        scores = M_k.sum(dim=0)
        top10  = torch.topk(scores, min(10, n_docs)).indices.cpu().tolist()

        torch.cuda.synchronize()
        score_ms = (time.perf_counter() - t0) * 1000.0
        kmeans_latency.add_ratio(r, score_ms)
        # ────────────────────────────────────────────────────────────

        key = f"kmeans_r{int(r*100)}"
        m   = hit_metrics(top10, gt_set)
        record(kmeans_metrics, kmeans_domain_metrics, key, m, domain)
        query_row.update({
            f'{key}_r@1':     m['r1'],
            f'{key}_r@5':     m['r5'],
            f'{key}_r@10':    m['r10'],
            f'{key}_ndcg@10': round(m['n10'], 4),
        })

    kmeans_query_rows.append(query_row)

print_summary(kmeans_metrics, kmeans_domain_metrics, METHOD_KEYS_KMEANS,
              title="Spherical KMeans Pooling Results")
kmeans_latency.report()

pd.DataFrame(kmeans_query_rows).to_csv(
    os.path.join(WORKING_DIR, "kmeans_queries.csv"), index=False)
print("\n✅ Saved: kmeans_queries.csv")

In [ ]:
# ==============================================================================
# METHOD 6 — Random Token Pruning (v14)
#
# Ablation baseline: instead of using any importance signal, tokens are sampled
# UNIFORMLY AT RANDOM.  If random pruning performs comparably to attention-based
# or SVD-based pruning, the importance signal offers little value.
#
# For each (query, ratio) pair: sample round(N * ratio) content tokens
# N_RANDOM_SEEDS times and average the resulting MaxSim scores to reduce
# variance from the random draw.
#
# Queries are encoded live with the plain forward pass (no attentions needed).
# ==============================================================================

print(">>> METHOD 6: Random Token Pruning (Ablation Baseline)")

# ---------- Keys ----------
METHOD_KEYS_RAND = ['traditional'] + [f"rand_r{int(r*100)}" for r in TOPK_RATIOS]

rand_metrics        = {}
rand_domain_metrics = {}
rand_query_rows     = []
rand_latency        = LatencyTracker("Random Pruning")

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Random Pruning")

for q_idx, item in pbar:
    question = item['question']
    gt_set   = set(item['gt_embed_indices'])
    domain   = item['domain']

    # ── Encode query live (plain forward — no attentions needed) ──────────
    q_inputs = query_processor.process_queries([question]).to(device)

    with torch.no_grad():
        q_proj = query_model(**q_inputs)   # (1, S, dim)

    attn_mask = q_inputs['attention_mask'][0]
    trad_idx  = torch.where(attn_mask > 0)[0]
    q_emb     = q_proj[0][trad_idx].float()
    q_norm    = F.normalize(q_emb, dim=-1)   # (N, D) — content tokens only

    # ── Baseline (full tokens) ────────────────────────────────────────────
    M_full     = fast_maxsim(q_norm, doc_matrix, doc_mask)
    trad_sc    = M_full.sum(dim=0)
    top10_trad = torch.topk(trad_sc, min(10, n_docs)).indices.cpu().tolist()
    m_trad     = hit_metrics(top10_trad, gt_set)
    record(rand_metrics, rand_domain_metrics, 'traditional', m_trad, domain)

    query_row = {
        'query_id': q_idx, 'doc_name': item['doc_name'],
        'domain': domain,  'question': question,
        'trad_r@1': m_trad['r1'],   'trad_r@5': m_trad['r5'],
        'trad_r@10': m_trad['r10'], 'trad_ndcg@10': round(m_trad['n10'], 4),
        'n_random_seeds': N_RANDOM_SEEDS,
    }

    # ── Random pruning — all ratios, averaged over N_RANDOM_SEEDS ─────────
    # Timing is measured inside random_prune_topk per ratio, covering:
    # random sampling + MaxSim + sum  (the actual scoring work).
    ratio_scores, ratio_timing = random_prune_topk(
        q_norm      = q_norm,
        content_idx = trad_idx,          # passed for API symmetry; not used internally
        doc_matrix  = doc_matrix,
        doc_mask    = doc_mask,
        topk_ratios = TOPK_RATIOS,
        n_seeds     = N_RANDOM_SEEDS,
    )

    for r in TOPK_RATIOS:
        key    = f"rand_r{int(r*100)}"
        scores = ratio_scores[r]
        rand_latency.add_ratio(r, ratio_timing[r])

        top10 = torch.topk(scores, min(10, n_docs)).indices.cpu().tolist()
        m = hit_metrics(top10, gt_set)
        record(rand_metrics, rand_domain_metrics, key, m, domain)
        query_row.update({
            f'{key}_r@1':     m['r1'],
            f'{key}_r@5':     m['r5'],
            f'{key}_r@10':    m['r10'],
            f'{key}_ndcg@10': round(m['n10'], 4),
        })

    rand_query_rows.append(query_row)

print_summary(rand_metrics, rand_domain_metrics, METHOD_KEYS_RAND,
              title="Random Token Pruning Results")
rand_latency.report()

pd.DataFrame(rand_query_rows).to_csv(
    os.path.join(WORKING_DIR, "random_pruning_queries.csv"), index=False)
print("\n✅ Saved: random_pruning_queries.csv")

In [ ]:
# ==============================================================================
# FINAL SUMMARY — aggregate results from all methods
# ==============================================================================

print("=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

# ---- Traditional ----
print_summary(trad_metrics, trad_domain_metrics, ['traditional'],
              title="Traditional MaxSim")

# ---- Hierarchical best (show all ratios) ----
print_summary(hier_metrics, hier_domain_metrics, METHOD_KEYS_HIER,
              title="Hierarchical Ward Pooling")

# ---- Ours (show traditional and trad_weighted) ----
print_summary(ours_metrics, ours_domain_metrics, ['traditional', 'trad_weighted'],
              title="Ours — Baseline rows (add importance pkl for full ablation)")

# ---- Attention (L=1, trad scoring) ----
print_summary(attn_metrics, attn_domain_metrics,
              ['traditional'] + [f"attn_L1_r{int(r*100)}_trad" for r in TOPK_RATIOS],
              title="Attention Score Pruning (L=1)")

# ---- Spherical KMeans ----
print_summary(kmeans_metrics, kmeans_domain_metrics, METHOD_KEYS_KMEANS,
              title="Spherical KMeans Pooling")

# ---- Random Pruning ----
print_summary(rand_metrics, rand_domain_metrics, METHOD_KEYS_RAND,
              title="Random Token Pruning (Ablation Baseline)")

# ==============================================================================
# Save master summary CSVs
# ==============================================================================

def save_summary_csv(metrics, domain_metrics, method_keys, prefix):
    # Overall
    rows = []
    for key in method_keys:
        if key not in metrics: continue
        m = metrics[key]; cnt = m['count'] or 1
        rows.append({
            'method':   key,
            'r@1':      round(m['r1']  / cnt * 100, 4),
            'r@5':      round(m['r5']  / cnt * 100, 4),
            'r@10':     round(m['r10'] / cnt * 100, 4),
            'ndcg@1':   round(m['n1']  / cnt,       6),
            'ndcg@5':   round(m['n5']  / cnt,       6),
            'ndcg@10':  round(m['n10'] / cnt,       6),
            'count':    cnt,
        })
    df_sum = pd.DataFrame(rows)
    df_sum.to_csv(os.path.join(WORKING_DIR, f"{prefix}_summary.csv"), index=False)

    # Per-domain
    dom_rows = []
    for domain in sorted(domain_metrics):
        dm  = domain_metrics[domain]
        row = {'domain': domain}
        for key in method_keys:
            m_   = dm.get(key, _init_metric()); cnt_ = m_['count'] or 1
            row[f'{key}_ndcg10'] = round(m_['n10'] / cnt_, 6)
            row[f'{key}_r10']    = round(m_['r10'] / cnt_ * 100, 4)
        dom_rows.append(row)
    pd.DataFrame(dom_rows).to_csv(
        os.path.join(WORKING_DIR, f"{prefix}_domain.csv"), index=False)

    print(f"✅ Saved: {prefix}_summary.csv and {prefix}_domain.csv")

save_summary_csv(trad_metrics, trad_domain_metrics, ['traditional'], "traditional")
save_summary_csv(hier_metrics, hier_domain_metrics, METHOD_KEYS_HIER, "hierarchical")
save_summary_csv(ours_metrics, ours_domain_metrics, ABLATION_KEYS_OURS, "ours_ablation")
save_summary_csv(attn_metrics, attn_domain_metrics, METHOD_KEYS_ATTN, "attention_pruning")
save_summary_csv(kmeans_metrics, kmeans_domain_metrics, METHOD_KEYS_KMEANS, "spherical_kmeans")
save_summary_csv(rand_metrics, rand_domain_metrics, METHOD_KEYS_RAND, "random_pruning")

print("\n>>> All evaluations complete.")